# Joshi Part 7: Binomial Trees & Implied Volatility (Rust)

Using `barter_joshi::trees` and `barter_joshi::solvers`.

**Kernel**: evcxr (Rust Jupyter kernel)

In [ ]:
:dep barter-joshi = { path = "../../barter-joshi" }

## 1. Binomial Tree Convergence

In [ ]:
use barter_joshi::payoff::{Call, Put};
use barter_joshi::trees::*;

let call = Call::new(100.0);
let put = Put::new(100.0);

println!("{:>6} {:>12} {:>12}", "Steps", "Call", "Put");
println!("{:-<32}", "");
for steps in [10, 50, 100, 200, 500] {
    let c = binomial_european(&call, 100.0, 0.05, 0.2, 1.0, steps);
    let p = binomial_european(&put, 100.0, 0.05, 0.2, 1.0, steps);
    println!("{:>6} {:>12.4} {:>12.4}", steps, c, p);
}
println!("\nBS exact: Call=10.4506, Put=5.5735");

## 2. American vs European Put

In [ ]:
let put = Put::new(100.0);
let euro = binomial_european(&put, 100.0, 0.05, 0.2, 1.0, 500);
let amer = binomial_american(&put, 100.0, 0.05, 0.2, 1.0, 500);

println!("European Put: {euro:.4}");
println!("American Put: {amer:.4}");
println!("Early exercise premium: {:.4}", amer - euro);

## 3. Implied Volatility

In [ ]:
use barter_joshi::black_scholes;
use barter_joshi::solvers::*;

// Round-trip: price at vol=25%, then recover
let true_vol = 0.25;
let price = black_scholes::call_price(100.0, 100.0, 0.05, true_vol, 1.0);

let newton = implied_vol_call(price, 100.0, 100.0, 0.05, 1.0, 0.2, 1e-8, 100).unwrap();
let bisect = implied_vol_call_bisection(price, 100.0, 100.0, 0.05, 1.0, 1e-8, 100).unwrap();

println!("True vol:           {true_vol:.8}");
println!("Newton-Raphson:     {newton:.8}");
println!("Bisection:          {bisect:.8}");
println!("Newton error:       {:.2e}", (newton - true_vol).abs());
println!("Bisection error:    {:.2e}", (bisect - true_vol).abs());

## 4. Implied Vol Smile

In [ ]:
println!("{:>8} {:>10} {:>10} {:>10}", "Strike", "MktPrice", "TrueVol", "ImplVol");
println!("{:-<42}", "");

for strike in (80..=120).step_by(5) {
    let k = strike as f64;
    let true_vol = 0.20 + 0.001 * (100.0 - k); // simple skew
    let mkt_price = black_scholes::call_price(100.0, k, 0.05, true_vol, 1.0);
    let iv = implied_vol_call(mkt_price, 100.0, k, 0.05, 1.0, 0.2, 1e-8, 100).unwrap();
    println!("{:>8.1} {:>10.4} {:>10.4} {:>10.4}", k, mkt_price, true_vol, iv);
}